# 05. 팀 취합 단계 검증 (Team Verification)

README에 새로 들어간 수치(그룹 교차검증, 변수 묶음 제거, 불균형 처리 반복 비교, 6종 분류, Y 좌표 제거, 데이터 구조 확인)를 재현하는 노트북입니다.

- 모든 실험은 **Train 데이터(1,552건)** 안에서만 수행합니다. Test는 사용하지 않습니다.
- 최종 모델은 README의 하이퍼파라미터로 **재현**한 것입니다. 파생변수 10개의 계산식은 변수 이름과 설명을 바탕으로 다시 작성했기 때문에, 공식 모델과 점수가 소수점 둘째 ~ 셋째 자리에서 다를 수 있습니다. 비교는 항상 **이 노트북 안에서 같은 조건끼리** 합니다.
- 전체 실행 시간: 1코어 기준 약 1시간 (셀별 소요 시간은 각 셀 위에 적었습니다).

In [ ]:
import sys, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import kruskal
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler

df = pd.read_csv("../data/processed/Faults_processed.csv")
# 팀 공통 분할 재현 (src/make_split.py와 같은 설정: 8:2 층화, random_state=42)
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.2, stratify=df["target"], random_state=42)
print(len(tr), len(te))
print(df.iloc[tr]["target"].value_counts().to_dict())
# 기대값: Other_Faults 538, Bumps 321, K_Scratch 313, Z_Scratch 152, Pastry 126, Stains 58, Dirtiness 44

## 1. 데이터 사실 확인 (README 5.2, 6.2절)

In [ ]:
raw = pd.read_csv("../data/raw/Faults.csv")
print("결함 오른쪽 끝 <= Length_of_Conveyer 인 행 비율:", (raw.X_Maximum <= raw.Length_of_Conveyer).mean())
print("면적-밝기합 Spearman:", round(df[["Pixels_Areas","Sum_of_Luminosity"]].corr(method="spearman").iloc[0,1],3))
print("A300 + A400 값:", (raw.TypeOfSteel_A300 + raw.TypeOfSteel_A400).unique())
# 유사 중복 쌍과 분할 위치
for i in [1775, 1776, 529, 678]:
    print(i, "Train" if i in set(tr) else "Test")
display(df.loc[[1775,1776]].T)

In [ ]:
# 극단값 구성 (Train 기준, 면적 사분위 범위 1.5배 밖)
T = df.iloc[tr]
q1, q3 = T.Pixels_Areas.quantile([.25,.75])
m = T.Pixels_Areas > q3 + 1.5*(q3-q1)
print("극단값 수:", m.sum(), " 그중 K_Scratch 비율:", round((T.target[m]=="K_Scratch").mean(),3))
# Kruskal-Wallis (Train 기준)
kw = []
for c in T.columns.drop("target"):
    H, p = kruskal(*[T.loc[T.target==k, c] for k in T.target.unique()])
    kw.append((c, (H-6)/(len(T)-7), p))
display(pd.DataFrame(kw, columns=["변수","효과크기 ε²","p"]).sort_values("효과크기 ε²", ascending=False))

## 2. 생산 조건 묶음 구조 (README 6.1절 발견 2)

In [ ]:
df["grp"] = df.TypeOfSteel_A300.astype(str)+"_"+df.Steel_Plate_Thickness.astype(str)+"_"+df.Length_of_Conveyer.astype(str)
T, Te = df.iloc[tr], df.iloc[te]
def purity(d):
    n = d.groupby("grp").size()
    c = d.groupby("grp").target.agg(lambda s: s.value_counts().iloc[0]/len(s))
    return (c*n).sum()/n.sum()
rng = np.random.default_rng(0); sh=[]
for _ in range(200):
    t = T.copy(); t["target"] = rng.permutation(t.target.values); sh.append(purity(t))
n = T.groupby("grp").size()
print("묶음 수:", len(n), " 중앙값:", n.median(), " 최대:", n.max())
print("실제 순도:", round(purity(T),3), " 무작위 평균/최대:", round(np.mean(sh),3), round(np.max(sh),3))
print("Test 행 중 Train에 같은 묶음이 있는 비율:", round(Te.grp.isin(set(T.grp)).mean(),3))
print("Test 행 중 처음 보는 묶음:", (~Te.grp.isin(set(T.grp))).sum(), "건")

## 3. 모양만으로 구분되는 정도 (README 6.1절 발견 3)

In [ ]:
SHAPE = ["LogOfAreas","Log_X_Index","Log_Y_Index","Orientation_Index","Square_Index","Empty_Index",
         "Edges_X_Index","Edges_Y_Index","Luminosity_Index","Minimum_of_Luminosity","Maximum_of_Luminosity"]
PROC = ["TypeOfSteel_A300","Steel_Plate_Thickness","Length_of_Conveyer"]
lab = df.target.to_numpy()
def neighbor_ratio(cols):
    Z = StandardScaler().fit_transform(df[cols])
    _, ind = NearestNeighbors(n_neighbors=6).fit(Z).kneighbors(Z)
    return pd.Series((lab[ind[:,1:]] == lab[:,None]).mean(1)).groupby(lab).mean()
display(pd.DataFrame({"모양·밝기만": neighbor_ratio(SHAPE), "+ 공정 조건": neighbor_ratio(SHAPE+PROC)}).round(3))

## 4. 롤 마크 주기 (README 6.1절 발견 4) — 약 2분

In [ ]:
df["xc"] = (df.X_Minimum + df.X_Maximum)/2
col = df[(df.Length_of_Conveyer==1358)&(df.Steel_Plate_Thickness==40)&(df.TypeOfSteel_A300==0)&(abs(df.xc-125)<=20)]
y = col.Y_Minimum.values
def fit_rate(yy, p, tol=15):
    d = np.diff(np.sort(yy)); d = d[d>0]; r = d/p
    return np.mean(np.abs(r-np.round(r))*p <= tol)
best_s, best_p = max((fit_rate(y,p), p) for p in np.arange(1000,20000,0.5))
print(len(col), col.target.value_counts().to_dict(), " 최적 주기:", best_p, " 적합률:", round(best_s,3))
rng = np.random.default_rng(0)
fixed = np.mean([fit_rate(rng.uniform(y.min(),y.max(),len(y)), best_p) for _ in range(200)])
search = np.mean([max(fit_rate(yy,p) for p in np.arange(1000,20000,2))
                  for yy in [rng.uniform(y.min(),y.max(),len(y)) for _ in range(20)]])
print("무작위(같은 주기):", round(fixed,3), " 무작위(주기 다시 탐색):", round(search,3))
print("Train/Test 분포:", pd.Series(["Train" if i in set(tr) else "Test" for i in col.index]).value_counts().to_dict())

## 5. 모델 정의 (최종 모델 재현 + Baseline)

In [ ]:
D = df.iloc[tr].reset_index(drop=True)
classes = np.unique(D.target); yi = np.searchsorted(classes, D.target.values)
X0 = D.drop(columns=["target","grp","xc"])
grp = D.grp.values

def add_features(X, Xfit):
    X = X.copy(); e = 1e-9
    X["compactness"] = X.Pixels_Areas/((X.X_Perimeter+X.Y_Perimeter)**2+e)
    X["perimeter_ratio_log"] = np.log1p(X.X_Perimeter)-np.log1p(X.Y_Perimeter)
    X["x_straightness"] = X.X_Perimeter/(X.X_Maximum-X.X_Minimum+1)
    X["lum_range"] = X.Maximum_of_Luminosity-X.Minimum_of_Luminosity
    X["lum_contrast"] = X.lum_range/(X.Maximum_of_Luminosity+X.Minimum_of_Luminosity+e)
    mean = X.Sum_of_Luminosity/X.Pixels_Areas
    X["lum_mean_position"] = (mean-X.Minimum_of_Luminosity)/(X.lum_range+e)
    xc = (X.X_Minimum+X.X_Maximum)/2
    X["x_center_rel"] = xc/X.Length_of_Conveyer
    X["x_center_dist"] = (X.x_center_rel-0.5).abs()
    key = lambda Z: Z.TypeOfSteel_A300.astype(str)+"_"+Z.Steel_Plate_Thickness.astype(str)+"_"+Z.Length_of_Conveyer.astype(str)
    X["plate_defect_count"] = key(X).map(key(Xfit).value_counts()).fillna(0).values   # 학습 fold로만 집계
    X["thickness_per_conveyer"] = X.Steel_Plate_Thickness/X.Length_of_Conveyer
    return X

class FinalModel:
    """ET + RF + GBM 소프트보팅 (README 8.1절 설정)"""
    def __init__(self, fe=True, drop_proc=False, only_proc=False, drop_pdc=False):
        self.fe, self.drop_proc, self.only_proc, self.drop_pdc = fe, drop_proc, only_proc, drop_pdc
    def _prep(self, X):
        Z = add_features(X, self.Xfit) if self.fe else X.copy()
        if self.drop_proc: Z = Z.drop(columns=[c for c in PROC+["plate_defect_count","thickness_per_conveyer"] if c in Z])
        if self.only_proc: Z = Z[PROC]
        if self.drop_pdc: Z = Z.drop(columns=["plate_defect_count"])
        return Z
    def fit(self, X, y):
        self.Xfit = X; Z = self._prep(X)
        self.models = [ExtraTreesClassifier(438, max_depth=15, max_features=0.7, class_weight="balanced", random_state=42),
                       RandomForestClassifier(483, max_features=0.3, random_state=42),
                       GradientBoostingClassifier(n_estimators=240, learning_rate=0.0879, max_depth=4, subsample=0.9079,
                                                  max_features=0.5, min_samples_leaf=5, random_state=42)]
        self.models[0].fit(Z, y); self.models[1].fit(Z, y)
        self.models[2].fit(Z, y, sample_weight=compute_sample_weight("balanced", y))
        return self
    def predict_proba(self, X):
        Z = self._prep(X); return np.mean([m.predict_proba(Z) for m in self.models], axis=0)

class XGBBase:
    def fit(self, X, y): self.m = XGBClassifier(random_state=42, eval_metric="mlogloss").fit(X, y); return self
    def predict_proba(self, X): return self.m.predict_proba(X)

class GBXGB_BSMOTE:
    """GB + XGBoost 소프트보팅, 각 모델 학습 fold 안에서 BorderlineSMOTE"""
    def fit(self, X, y):
        self.ms = []
        for mk in [lambda: GradientBoostingClassifier(random_state=42),
                   lambda: XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6, eval_metric="mlogloss", random_state=42)]:
            Xa, ya = BorderlineSMOTE(k_neighbors=3, m_neighbors=5, random_state=42).fit_resample(X.values, y)
            self.ms.append(mk().fit(Xa, ya))
        return self
    def predict_proba(self, X): return np.mean([m.predict_proba(X.values) for m in self.ms], axis=0)

def run_cv(make, kind="strat", seeds=(42,0,1), X=None, y=None, groups=None):
    X = X0 if X is None else X; y = yi if y is None else y; groups = grp if groups is None else groups
    scores = []
    for s in seeds:
        cv = StratifiedKFold(5, shuffle=True, random_state=s) if kind=="strat" else StratifiedGroupKFold(5, shuffle=True, random_state=s)
        for a, b in cv.split(X, y, groups if kind=="group" else None):
            p = make().fit(X.iloc[a], y[a]).predict_proba(X.iloc[b]).argmax(1)
            scores.append(f1_score(y[b], p, average="macro"))
    return np.array(scores)

## 6. 일반 교차검증 vs 그룹 교차검증 (README 7.4, 7.5, 8.4절) — 약 40분

In [ ]:
variants = {
    "Baseline XGBoost": XGBBase,
    "최종 모델": lambda: FinalModel(),
    "최종 모델 - 파생변수 제외": lambda: FinalModel(fe=False),
    "최종 모델 - 공정 조건 제외": lambda: FinalModel(drop_proc=True),
    "공정 조건 3개만": lambda: FinalModel(fe=False, only_proc=True),
}
res = {}
for name, mk in variants.items():
    t = time.time()
    res[name] = {k: run_cv(mk, k) for k in ["strat","group"]}
    print(f"{name:25s} 일반 {res[name]['strat'].mean():.4f}±{res[name]['strat'].std():.4f}  "
          f"그룹 {res[name]['group'].mean():.4f}±{res[name]['group'].std():.4f}  ({time.time()-t:.0f}초)")
# 같은 fold끼리 짝 비교
for a, b in [("최종 모델","Baseline XGBoost"), ("최종 모델","최종 모델 - 파생변수 제외")]:
    for k in ["strat","group"]:
        d = res[a][k]-res[b][k]
        print(f"{a} - {b} [{k}]: 평균 {d.mean():+.4f}, 15번 중 {(d>0).sum()}번 우세")

In [ ]:
# plate_defect_count만 뺀 경우 (시드 42 한 번), 윤하 후보의 그룹 CV
print("파생 9개(plate_defect_count 제외):",
      run_cv(lambda: FinalModel(drop_pdc=True), "strat", (42,)).mean().round(4),
      run_cv(lambda: FinalModel(drop_pdc=True), "group", (42,)).mean().round(4))
g = run_cv(GBXGB_BSMOTE, "group")
print("GB+XGB+BSMOTE 그룹 CV: 시드42", g[:5].mean().round(4), " 반복", g.mean().round(4), "±", g.std().round(4))

## 7. 불균형 처리 반복 비교 (README 7.3절) — 약 15분

In [ ]:
Xv = X0.values
def fit_gb(method, Xa, ya):
    sw = None
    if method == "클래스 가중치": sw = compute_sample_weight("balanced", ya)
    elif method == "SMOTE": Xa, ya = SMOTE(k_neighbors=3, random_state=42).fit_resample(Xa, ya)
    elif method == "BorderlineSMOTE": Xa, ya = BorderlineSMOTE(k_neighbors=3, m_neighbors=5, random_state=42).fit_resample(Xa, ya)
    elif method == "언더샘플링": Xa, ya = RandomUnderSampler(random_state=42).fit_resample(Xa, ya)
    return GradientBoostingClassifier(random_state=42).fit(Xa, ya, sample_weight=sw)
imb = {}
for meth in ["처리 없음","클래스 가중치","BorderlineSMOTE","SMOTE","언더샘플링"]:
    sc = []
    for s in [42,0,1]:
        for a, b in StratifiedKFold(5, shuffle=True, random_state=s).split(Xv, yi):
            sc.append(f1_score(yi[b], fit_gb(meth, Xv[a], yi[a]).predict(Xv[b]), average="macro"))
    imb[meth] = np.array(sc); d = imb[meth]-imb["처리 없음"]
    print(f"{meth:15s} {imb[meth].mean():.4f}±{imb[meth].std():.4f}  차이 {d.mean():+.4f}  우세 {(d>0).sum()}/15")

## 8. 정답 체계와 Y 좌표 (README 8.2, 8.4절) — 약 5분

In [ ]:
keep = D.target.values != "Other_Faults"
X6 = X0[keep].reset_index(drop=True); y6 = np.searchsorted(np.unique(yi[keep]), yi[keep])
print("기타 제외 6종:", run_cv(lambda: FinalModel(), "strat", (42,), X=X6, y=y6).mean().round(4))
XnoY = X0.drop(columns=["Y_Minimum","Y_Maximum"])
for k in ["strat","group"]:
    print("Y 제외", k, run_cv(lambda: FinalModel(), k, (42,), X=XnoY).mean().round(4),
          " / Y 포함", k, run_cv(lambda: FinalModel(), k, (42,)).mean().round(4))